#### Demo3 CAST Project

In [6]:
import CAST
import scanpy as sc
import os
import numpy as np
import warnings
import dgl
import torch
import CAST
import os
import numpy as np
import anndata as ad
import scanpy as sc
import warnings
import pandas as pd
warnings.filterwarnings("ignore")
work_dir = '/p1/data/jtian/SA/05_CAST' #### input the demo path

In [2]:
output_path = f'{work_dir}/output_demo3_CAST_project'
os.makedirs(output_path,exist_ok=True)
graph_list = {'rep1':['15','0']} # source_sample, target_sample

#### CAST Project to project the STARmap data to the RIBOmap data

In [ ]:
#UMAP+PCA
sc.tl.pca(sdata, svd_solver='full', n_comps=50)

# 3) 邻居图 + UMAP（用 PCA）
sc.pp.neighbors(sdata, n_neighbors=50, n_pcs=30, use_rep='X_pca')
sc.tl.umap(sdata, min_dist=0.01, spread=5)

In [3]:
import scanpy as sc
import numpy as np
import torch

# 读数据
sdata = sc.read_h5ad("/p1/data/jtian/SA/05_CAST/output_demo3_CAST_project/demo1_cast9.h5ad")

# ========= 1) 表达层准备（你现在没有 raw，就先用 X 作为占位） =========
# 注意：如果 sdata.X 已经是 log+scale，这里的 raw 只是“伪raw”，可以跑但不要当真实表达解释
sdata.layers['raw'] = sdata.X.copy()
sdata.layers['log1p_norm'] = sdata.X.copy()
sdata.layers['log1p_norm_scaled'] = sdata.X.copy()

# ========= 2) 读取对齐坐标 =========
coords = torch.load("/p1/data/jtian/SA/05_CAST/output_cast9/15_align_to_0_coords_final.data", map_location='cpu')

# 统一把 coords[key] 转成 numpy（兼容 torch.Tensor / numpy.ndarray）
def to_numpy(a):
    if isinstance(a, torch.Tensor):
        return a.detach().cpu().numpy()
    return np.asarray(a)

c15 = to_numpy(coords['15'])
c0  = to_numpy(coords['0'])

mask15 = sdata.obs['sample'].astype(str) == '15'
mask0  = sdata.obs['sample'].astype(str) == '0'

# 确保列存在
if 'x' not in sdata.obs.columns:
    sdata.obs['x'] = np.nan
if 'y' not in sdata.obs.columns:
    sdata.obs['y'] = np.nan

# ✅ 强烈建议：先做数量一致性检查（不一致就不要写入）
assert c15.shape[0] == mask15.sum(), f"coords['15']行数={c15.shape[0]} 但 sample=15 的细胞数={mask15.sum()}"
assert c0.shape[0]  == mask0.sum(),  f"coords['0']行数={c0.shape[0]} 但 sample=0 的细胞数={mask0.sum()}"

# 写入对齐坐标
sdata.obs.loc[mask15, 'x'] = c15[:, 0]
sdata.obs.loc[mask15, 'y'] = c15[:, 1]

sdata.obs.loc[mask0, 'x'] = c0[:, 0]
sdata.obs.loc[mask0, 'y'] = c0[:, 1]

# ========= 3) CAST preprocess =========
sdata = CAST.preprocess_fast(sdata, mode='default')
batch_key = 'sample'


Preprocessing...


The `CAST_PROJECT()` function is used to project the `source_sample` (query sample) to `target_sample` (reference sample).


In [5]:
sdata_refs = {}
list_ts = {}
color_dict = {
        'TEPN': '#256b00',
        'INH': '#ee750a',
        'CHO_PEP': '#f280cf',
        'DE_MEN': '#f24f4b',
        'AC': '#e8e879',
        'OLG': '#a8e1eb',
        'VAS': '#395ba8',
        'CHOR_EPEN': '#697491',
        'PVM': '#8803fc',
        'MLG': '#23ccb8',
        'OPC': '#667872',
        'Other': '#ebebeb'
    }
for rep in graph_list.keys():
    print(f'Start the {rep} samples:')
    source_sample, target_sample = graph_list[rep]
    output_path_t = f'{output_path}/{source_sample}_to_{target_sample}'
    os.makedirs(output_path_t,exist_ok=True)
    sdata_refs[rep],list_ts[rep] = CAST.CAST_PROJECT(
        sdata_inte = sdata[np.isin(sdata.obs[batch_key],[source_sample, target_sample])], # the integrated dataset
        source_sample = source_sample, # the source sample name
        target_sample = target_sample, # the target sample name
        coords_source = np.array(sdata[np.isin(sdata.obs[batch_key],source_sample),:].obs.loc[:,['x','y']]), # the coordinates of the source sample
        coords_target = np.array(sdata[np.isin(sdata.obs[batch_key],target_sample),:].obs.loc[:,['x','y']]), # the coordinates of the target sample
        scaled_layer = 'log1p_norm_scaled', # the scaled layer name in `adata.layers`, which is used to be integrated
        batch_key = batch_key, # the column name of the samples in `obs`
        source_sample_ctype_col = 'cell_type', # the column name of the cell type in `obs`
        output_path = output_path_t, # the output path
        integration_strategy = 'Harmony', # 'Harmony' or None (use existing integrated features)
        color_dict = color_dict, # the color dict for the cell type
        use_highly_variable_t = False
    )

Start the rep1 samples:
Running PCA based on the layer log1p_norm_scaled:


2026-01-22 11:23:20,179 - harmonypy - INFO - Running Harmony (PyTorch on cuda)
2026-01-22 11:23:20,180 - harmonypy - INFO -   Parameters:
2026-01-22 11:23:20,181 - harmonypy - INFO -     max_iter_harmony: 10
2026-01-22 11:23:20,181 - harmonypy - INFO -     max_iter_kmeans: 20
2026-01-22 11:23:20,182 - harmonypy - INFO -     epsilon_cluster: 1e-05
2026-01-22 11:23:20,182 - harmonypy - INFO -     epsilon_harmony: 0.0001
2026-01-22 11:23:20,183 - harmonypy - INFO -     nclust: 100
2026-01-22 11:23:20,184 - harmonypy - INFO -     block_size: 0.05
2026-01-22 11:23:20,184 - harmonypy - INFO -     lamb: [1. 1.]
2026-01-22 11:23:20,185 - harmonypy - INFO -     theta: [2. 2.]
2026-01-22 11:23:20,185 - harmonypy - INFO -     sigma: [0.1 0.1 0.1 0.1 0.1]...
2026-01-22 11:23:20,186 - harmonypy - INFO -     verbose: True
2026-01-22 11:23:20,186 - harmonypy - INFO -     random_state: 0
2026-01-22 11:23:20,186 - harmonypy - INFO -   Data: 50 PCs × 100804 cells
2026-01-22 11:23:20,188 - harmonypy - IN

Running Harmony integration:


2026-01-22 11:23:24,719 - harmonypy - INFO - KMeans initialization complete.
2026-01-22 11:23:24,725 - harmonypy - INFO - Iteration 1 of 10
2026-01-22 11:23:25,023 - harmonypy - INFO - Iteration 2 of 10
2026-01-22 11:23:25,321 - harmonypy - INFO - Iteration 3 of 10
2026-01-22 11:23:25,619 - harmonypy - INFO - Iteration 4 of 10
2026-01-22 11:23:25,877 - harmonypy - INFO - Iteration 5 of 10
2026-01-22 11:23:26,052 - harmonypy - INFO - Iteration 6 of 10
2026-01-22 11:23:26,227 - harmonypy - INFO - Converged after 6 iterations


ValueError: Value passed for key 'X_pca_harmony' is of incorrect shape. Values of obsm must match dimensions ('obs',) of parent. Value had shape (50,) while it should have had (100804,).